# 05. Integrated Pollutant Model

This notebook runs the Study 3 pollutant equations together over a 40-year horizon. It uses one-year steps and fixed emissions scenarios.

In [ ]:
import copy
import pandas as pd
import matplotlib.pyplot as plt

from study_3_pollutant_dynamics.config import EnvironmentConfig
from study_3_pollutant_dynamics.environment import PollutantDynamicsEnvironment
from study_3_pollutant_dynamics.scenario_parameters import get_parameters, SCENARIO_SETTINGS
from study_3_pollutant_dynamics.pollutant_models import pm25_exposure

## Reference settings and scenario labels

PM2.5 starts from the CDC 2020 population-weighted regional exposure estimates. Its source-receptor shape follows EPA COBRA, but the regional transfer coefficients and cross-region shares are scenario assumptions. CO2 starts from NOAA's observed 2025 global concentration; its reference regional emissions are held at the EIA 2023 state-summed values. The observed global growth path already includes emissions from these regions, so only emissions deviations from that reference are added through the impulse-response function.

The nitrate settings use the partial-year Keokuk USGS pilot as a concentration and flow anchor. Its residence time, denitrification rate, and replicated regional catchments are assumptions. Lead starts from the USGS geogenic soil median in a synthetic 1-hectare, 0–5 cm soil layer. Its water-to-soil ratio and annual mobile export are assumptions.

In [2]:
reference_parameters = get_parameters(episode_steps=40)
reference_config = EnvironmentConfig(reference_parameters)

scenario_rows = []
for pollutant, scenario_name in reference_parameters["scenario_names"].items():
    settings = SCENARIO_SETTINGS[pollutant][scenario_name]
    row = {"pollutant": pollutant, "scenario": scenario_name}
    for parameter_name, parameter_value in settings.items():
        row[parameter_name] = parameter_value
    scenario_rows.append(row)

scenario_table = pd.DataFrame(scenario_rows)
display(scenario_table.fillna("").round(5))

,pollutant,scenario,transfer_ug_m3_per_1000_tons,cross_region_share,background_growth_ppm_year,emission_multiplier,input_multiplier,denitrification_rate_per_year,residence_time_years,soil_concentration_mg_kg,kd_l_per_kg,annual_input_kg_pb,annual_mobile_export_fraction
0,pm25,reference,0.02,0.2,,,,,,,,,
1,co2,reference,,,2.5,1.0,,,,,,,
2,nitrate,reference,,,,,1.0,0.1,0.5,,,,
3,lead,reference,,,,,,,,18.1,2587.0,0.1,0.01


## Run fixed emissions scenarios

The reference run holds the 2020 PM2.5 exposure baseline and EIA regional CO2 emissions reference constant. The increased-emissions run adds a hypothetical 5,000 metric tons per year of primary PM2.5 emission changes in each region, raises modeled regional CO2 emissions by 10%, increases nitrate input load by 20%, and increases the synthetic lead input to 0.5 kg/year per modeled soil parcel. 

In [ ]:
def run_environment(parameters, scenario_name):
    config = EnvironmentConfig(parameters)
    environment = PollutantDynamicsEnvironment(config)
    observations = environment.reset(seed=0)
    rows = []
    flow_rows = []

    for year in range(1, config.episode_steps + 1):
        observations, terminated, truncated, info = environment.step()
        pm_values = info["stocks"]["PM2.5"]
        nitrate_values = []
        lead_total_values = info["stocks"]["Lead"]
        lead_mobile_values = info["mobile_stocks"]["Lead"]

        for region in range(len(config.regions)):
            nitrate_mass = info["stocks"]["Nitrate-N"][region]
            nitrate_concentration = 1000 * nitrate_mass / config.nitrate["water_volume_m3"]
            nitrate_values.append(nitrate_concentration)
            flow_rows.append({"scenario": scenario_name, "year": year, "region": config.regions[region],
                              "nitrate_mass_balance_residual_kg": info["nitrate_flows"][region]["mass_balance_residual_kg_n"],
                              "lead_mass_balance_residual_kg": info["lead_flows"][region]["mass_balance_residual_kg_pb"]})

        rows.append({
            "scenario": scenario_name,
            "year": year,
            "mean_PM2.5_ug_m3": sum(pm_values) / len(pm_values),
            "CO2_ppm": info["stocks"]["CO2"],
            "mean_nitrate_mg_N_L": sum(nitrate_values) / len(nitrate_values),
            "mean_lead_total_kg": sum(lead_total_values) / len(lead_total_values),
            "mean_lead_mobile_kg": sum(lead_mobile_values) / len(lead_mobile_values),
        })

        if terminated or truncated:
            break

    return pd.DataFrame(rows), pd.DataFrame(flow_rows), info

In [ ]:
baseline_parameters = get_parameters(episode_steps=40)
stress_parameters = copy.deepcopy(baseline_parameters)
stress_parameters["pm25"]["annual_emission_change_tons"] = [5000.0, 5000.0, 5000.0, 5000.0]

In [ ]:
for region in range(len(stress_parameters["co2"]["annual_emissions_gtco2_year"])):
    reference_emission = stress_parameters["co2"]["reference_emissions_gtco2_year"][region]
    stress_parameters["co2"]["annual_emissions_gtco2_year"][region] = reference_emission * 1.10

for region in range(len(stress_parameters["nitrate"]["annual_input_kg_n"])):
    stress_parameters["nitrate"]["annual_input_kg_n"][region] *= 1.20
    stress_parameters["lead"]["annual_input_kg_pb"][region] = 0.5

baseline_trajectory, baseline_flows, baseline_info = run_environment(baseline_parameters, "reference")
stress_trajectory, stress_flows, stress_info = run_environment(stress_parameters, "increased emissions")
trajectory = pd.concat([baseline_trajectory, stress_trajectory], ignore_index=True)
flow_checks = pd.concat([baseline_flows, stress_flows], ignore_index=True)

In [ ]:
summary_rows = []
for scenario_name, scenario_trajectory in trajectory.groupby("scenario", sort=False):
    final_row = scenario_trajectory.iloc[-1]
    summary_rows.append({
        "scenario": scenario_name,
        "years": len(scenario_trajectory),
        "final_mean_PM2.5_ug_m3": final_row["mean_PM2.5_ug_m3"],
        "final_CO2_ppm": final_row["CO2_ppm"],
        "final_mean_nitrate_mg_N_L": final_row["mean_nitrate_mg_N_L"],
        "final_mean_lead_total_kg": final_row["mean_lead_total_kg"],
        "final_mean_lead_mobile_kg": final_row["mean_lead_mobile_kg"],
    })

display(pd.DataFrame(summary_rows).round(4))

The reference CO2 run should follow only the selected global background growth path because its regional emissions match the reference emissions. The stress run adds a response to the 10% regional deviation. PM2.5 responds to the changed annual emissions but does not accumulate as a stock. Nitrate approaches a steady concentration determined by load, flushing, volume, and denitrification. Lead persists because it has no decay term; only the assumed mobile export removes lead from the modeled soil parcel.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_settings = [("mean_PM2.5_ug_m3", "Mean annual PM2.5 exposure (µg/m³)"), ("CO2_ppm", "Global CO2 concentration (ppm)"),
                 ("mean_nitrate_mg_N_L", "Mean nitrate-N concentration (mg N/L)"),
                 ("mean_lead_total_kg", "Mean total soil lead (kg Pb)")]

for axis, setting in zip(axes.flat, plot_settings):
    metric_name, y_label = setting
    for scenario_name, scenario_trajectory in trajectory.groupby("scenario", sort=False):
        axis.plot(scenario_trajectory["year"], scenario_trajectory[metric_name], label=scenario_name)
    axis.set_xlabel("Year")
    axis.set_ylabel(y_label)
    axis.grid(alpha=0.3)

axes[0, 0].legend()
figure.suptitle("Annual pollutant trajectories under fixed emissions")
plt.tight_layout()
plt.show()

## Sensitivity to uncertain assumptions

Each factor is varied independently across its low, reference, and high values while all other settings remain at reference. PM2.5 uses a fixed hypothetical 5,000-ton/year emissions change in each region. The table reports the relevant 40-year endpoint in its physical unit.

In [ ]:
sensitivity_factors = [
    {"pollutant": "PM2.5", "factor": "transfer coefficient", "key": "transfer_ug_m3_per_1000_tons", "metric": "mean_PM2.5_ug_m3"},
    {"pollutant": "PM2.5", "factor": "cross-region share", "key": "cross_region_share", "metric": "mean_PM2.5_ug_m3"},
    {"pollutant": "CO2", "factor": "background growth", "key": "background_growth_ppm_year", "metric": "CO2_ppm"},
    {"pollutant": "CO2", "factor": "regional emission multiplier", "key": "emission_multiplier", "metric": "CO2_ppm"},
    {"pollutant": "Nitrate-N", "factor": "annual input multiplier", "key": "input_multiplier", "metric": "mean_nitrate_mg_N_L"},
    {"pollutant": "Nitrate-N", "factor": "denitrification rate per year", "key": "denitrification_rate_per_year", "metric": "mean_nitrate_mg_N_L"},
    {"pollutant": "Nitrate-N", "factor": "residence time in years", "key": "residence_time_years", "metric": "mean_nitrate_mg_N_L"},
    {"pollutant": "Lead", "factor": "initial soil concentration", "key": "soil_concentration_mg_kg", "metric": "mean_lead_total_kg"},
    {"pollutant": "Lead", "factor": "partition coefficient Kd", "key": "kd_l_per_kg", "metric": "mean_lead_mobile_kg"},
    {"pollutant": "Lead", "factor": "annual input", "key": "annual_input_kg_pb", "metric": "mean_lead_total_kg"},
    {"pollutant": "Lead", "factor": "annual mobile export fraction", "key": "annual_mobile_export_fraction", "metric":"mean_lead_total_kg"},
]
scenario_levels = ("low", "reference", "high")

In [ ]:
sensitivity_rows = []
pollutant_keys = {"PM2.5": "pm25", "CO2": "co2", "Nitrate-N": "nitrate", "Lead": "lead"}

for factor in sensitivity_factors:
    pollutant_key = pollutant_keys[factor["pollutant"]]

    for scenario_level in scenario_levels:
        parameters = copy.deepcopy(baseline_parameters)
        scenario_value = SCENARIO_SETTINGS[pollutant_key][scenario_level][factor["key"]]

        if factor["pollutant"] == "PM2.5":
            parameters["pm25"][factor["key"]] = scenario_value
            parameters["pm25"]["annual_emission_change_tons"] = [5000.0, 5000.0, 5000.0, 5000.0]

        elif factor["pollutant"] == "CO2":
            if factor["key"] == "emission_multiplier":
                annual_emissions = []

                for value in parameters["co2"]["reference_emissions_gtco2_year"]:
                    annual_emissions.append(value * scenario_value)

                parameters["co2"]["annual_emissions_gtco2_year"] = annual_emissions

            else:
                parameters["co2"][factor["key"]] = scenario_value

        elif factor["pollutant"] == "Nitrate-N":
            if factor["key"] == "input_multiplier":
                annual_inputs = []
                reference_multiplier = SCENARIO_SETTINGS["nitrate"]["reference"]["input_multiplier"]

                for value in parameters["nitrate"]["annual_input_kg_n"]:
                    annual_inputs.append(value * scenario_value / reference_multiplier)

                parameters["nitrate"]["annual_input_kg_n"] = annual_inputs

            elif factor["key"] == "residence_time_years":
                annual_discharge = parameters["nitrate"]["annual_discharge_m3"]
                water_volume = annual_discharge * scenario_value

                parameters["nitrate"]["water_volume_m3"] = water_volume

                initial_concentration = parameters["nitrate"]["initial_concentration_mg_n_l"]
                initial_stock = initial_concentration * water_volume / 1000

                parameters["nitrate"]["initial_stock_kg_n"] = [initial_stock] * len(parameters["regions"])

            else:
                parameters["nitrate"][factor["key"]] = scenario_value

        elif factor["pollutant"] == "Lead":
            if factor["key"] == "soil_concentration_mg_kg":
                soil_mass = parameters["lead"]["soil_mass_kg"]
                initial_stock = scenario_value * soil_mass / 1000000

                parameters["lead"]["initial_stock_kg_pb"] = [initial_stock] * len(parameters["regions"])

            elif factor["key"] == "kd_l_per_kg":
                ratio = parameters["lead"]["water_to_soil_ratio_l_per_kg"]
                parameters["lead"]["mobile_fraction"] = ratio / (ratio + scenario_value)

            elif factor["key"] == "annual_input_kg_pb":
                parameters["lead"]["annual_input_kg_pb"] = [scenario_value] * len(parameters["regions"])

            else:
                parameters["lead"][factor["key"]] = scenario_value

        result, flow_result, final_info = run_environment(parameters, f"{factor['factor']}_{scenario_level}")
        final_value = result.iloc[-1][factor["metric"]]

        row = {
            "pollutant": factor["pollutant"],
            "factor": factor["factor"],
            "level": scenario_level,
            "parameter_value": scenario_value,
            "endpoint_metric": factor["metric"],
            "endpoint_value": final_value,
        }

        sensitivity_rows.append(row)


In [ ]:
sensitivity_table = pd.DataFrame(sensitivity_rows)
reference_values = sensitivity_table[sensitivity_table["level"] == "reference"].set_index("factor")["endpoint_value"]
sensitivity_table["change_from_reference"] = [
    row["endpoint_value"] - reference_values[row["factor"]]
    for _, row in sensitivity_table.iterrows()
]
display(sensitivity_table.round(5))

These are one-factor-at-a-time comparisons: each row changes one parameter from its reference value, then reports the change in the corresponding 40-year endpoint. The nitrate residence-time case changes the implied water volume and adjusts initial mass to preserve the starting concentration. The lead partition-coefficient case changes the mobile fraction while keeping total initial soil lead fixed. Parameter ranges are scenario assumptions, so the endpoint differences measure model sensitivity to those choices, not empirical uncertainty.

## Mathematical checks

These checks cover bookkeeping, nonnegative values, and expected responses to the controlled scenarios. They do not validate the scenario values against a complete set of regional environmental observations.

In [ ]:
check_rows = []

zero_change = pm25_exposure(baseline_parameters["pm25"]["baseline_exposure_ug_m3"], [0.0, 0.0, 0.0, 0.0],
                            baseline_parameters["pm25"]["transfer_ug_m3_per_1000_tons"],
                            baseline_parameters["pm25"]["cross_region_share"])
pm25_residuals = []
for region in range(len(zero_change)):
    residual = abs(zero_change[region] - baseline_parameters["pm25"]["baseline_exposure_ug_m3"][region])
    pm25_residuals.append(residual)
pm25_maximum_residual = max(pm25_residuals)
pm25_preserves_baseline = pm25_maximum_residual < 1e-12

In [ ]:
reference_co2_no_deviation = abs(baseline_info["co2_emission_deviation_gtco2"]) < 1e-12
maximum_nitrate_residual = flow_checks["nitrate_mass_balance_residual_kg"].abs().max()
maximum_lead_residual = flow_checks["lead_mass_balance_residual_kg"].abs().max()
nitrate_mass_balanced = maximum_nitrate_residual < 1e-5
lead_mass_balanced = maximum_lead_residual < 1e-10
nonnegative_outputs = True
nonnegative_columns = ["mean_PM2.5_ug_m3", "CO2_ppm", "mean_nitrate_mg_N_L", "mean_lead_total_kg", "mean_lead_mobile_kg"]

for column_name in nonnegative_columns:
    if (trajectory[column_name] < 0).any():
        nonnegative_outputs = False

check_rows.append({"check": "Zero PM2.5 emission change preserves baseline", "passed": pm25_preserves_baseline,
                   "maximum_residual":pm25_maximum_residual})

check_rows.append({"check": "Reference CO2 emissions have zero deviation", "passed": reference_co2_no_deviation,
                   "maximum_residual": abs(baseline_info["co2_emission_deviation_gtco2"])})

check_rows.append({"check": "Nitrate annual mass balance", "passed": nitrate_mass_balanced,
                   "maximum_residual": maximum_nitrate_residual})

check_rows.append({"check": "Lead annual mass balance", "passed": lead_mass_balanced, "maximum_residual": maximum_lead_residual})
check_rows.append({"check": "All simulated stocks and concentrations are nonnegative", "passed": nonnegative_outputs,
                   "maximum_residual": 0.0})

In [ ]:
reference_nitrate_stability_error = abs(baseline_trajectory.iloc[-1]["mean_nitrate_mg_N_L"]
                                        - baseline_parameters["nitrate"]["initial_concentration_mg_n_l"])

positive_emissions_raise_pm25 = (stress_trajectory["mean_PM2.5_ug_m3"].to_numpy() > baseline_trajectory["mean_PM2.5_ug_m3"].to_numpy()).all()

positive_co2_deviation_raises_concentration = (stress_trajectory["CO2_ppm"].to_numpy() > baseline_trajectory["CO2_ppm"].to_numpy()).all()

positive_nitrate_input_raises_concentration = (stress_trajectory["mean_nitrate_mg_N_L"].to_numpy() >
                                               baseline_trajectory["mean_nitrate_mg_N_L"].to_numpy()).all()

positive_lead_input_raises_total_stock = (stress_trajectory["mean_lead_total_kg"].to_numpy() >
                                          baseline_trajectory["mean_lead_total_kg"].to_numpy()).all()

reference_co2_increases_monotonically = baseline_trajectory["CO2_ppm"].is_monotonic_increasing

In [ ]:
check_rows.append({"check": "Reference nitrate remains at its defined equilibrium", "passed": reference_nitrate_stability_error < 1e-10,
                   "maximum_residual": reference_nitrate_stability_error})

check_rows.append({"check": "Higher PM2.5 emissions raise PM2.5 exposure", "passed": positive_emissions_raise_pm25,
                   "maximum_residual": 0.0})

check_rows.append({"check": "Positive regional CO2 deviations raise concentration",
                   "passed": positive_co2_deviation_raises_concentration, "maximum_residual": 0.0})

check_rows.append({"check": "Higher nitrate inputs raise nitrate concentration",
                   "passed": positive_nitrate_input_raises_concentration, "maximum_residual": 0.0})

check_rows.append({"check": "Higher lead inputs raise total soil lead", "passed": positive_lead_input_raises_total_stock,
                   "maximum_residual": 0.0})

check_rows.append({"check": "Reference CO2 concentration increases monotonically", "passed": reference_co2_increases_monotonically,
                   "maximum_residual": 0.0})

In [ ]:
check_table = pd.DataFrame(check_rows)
display(check_table)

checks_passed = True

for row in check_rows:
    if not row["passed"]:
        checks_passed = False

if checks_passed:
    print("All environmental validation checks passed.")
else:
    print("Review the failed checks before using these trajectories.")

## Summary and limits

The integrated model runs at one step per year and keeps PM2.5 concentration, CO2 concentration, nitrate-N mass/concentration, and total/mobile lead mass separate. The reference run is stable for nitrate because its input is calculated from the reference concentration, discharge, and denitrification rate. Lead accumulates slowly under a persistent input and does not decay. A PM2.5 emission change affects the annual concentration estimate without creating an artificial persistent air stock. CO2 follows its background reference path and responds to deviations in regional emissions.